# Local LLM Engineering

WebShop Pro Local AI Gateway: Ollama/OpenAI-compatible API, quantization döntések, lokális RAG, tool calling, benchmark, governance és hybrid routing.


In [ ]:
def estimate_model_ram(params_billion, bits_per_weight=4, context_tokens=8192):
    weights_gb = params_billion * 1_000_000_000 * bits_per_weight / 8 / 1e9
    kv_cache_gb = context_tokens / 8192 * params_billion * 0.10
    return round(weights_gb + kv_cache_gb + 1.2, 1)

for name, params in [("small", 3), ("daily", 8), ("reasoning", 14), ("server", 32)]:
    print(name, estimate_model_ram(params), "GB")


In [ ]:
from openai import OpenAI

local_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
# response = local_client.chat.completions.create(
#     model="llama3.1:8b",
#     messages=[{"role": "user", "content": "Mi a local LLM előnye?"}],
# )
# print(response.choices[0].message.content)
print("OpenAI-compatible local client configured")


In [ ]:
quantization = {
    "Q8_0": {"quality": 0.98, "ram": "high"},
    "Q6_K": {"quality": 0.96, "ram": "medium-high"},
    "Q5_K_M": {"quality": 0.94, "ram": "medium"},
    "Q4_K_M": {"quality": 0.90, "ram": "low"},
}
quantization


In [ ]:
from dataclasses import dataclass

@dataclass
class LLMConfig:
    name: str
    base_url: str
    api_key: str
    default_model: str
    timeout_s: int = 30

local_cfg = LLMConfig("local-ollama", "http://localhost:11434/v1", "ollama", "llama3.1:8b")
local_cfg


In [ ]:
def webshop_support_prompt(question, context):
    return f"""
Te a WebShop Pro lokális support asszisztense vagy.
Csak a CONTEXT alapján válaszolj.
Ha nincs elég információ, mondd: Erre nincs elég információm a lokális tudásbázisban.

CONTEXT:
{context}

USER QUESTION:
{question}
""".strip()

print(webshop_support_prompt("Mikor érkezik meg?", "Budapesten 2 munkanap."))


In [ ]:
documents = [
    {"id": "shipping-1", "source": "shipping_policy.md", "text": "Budapesten a standard szállítás 2 munkanap."},
    {"id": "returns-1", "source": "return_policy.md", "text": "A vásárló 14 napon belül jelezheti az elállást."},
    {"id": "warranty-1", "source": "warranty.md", "text": "Elektronikai termékekre 24 hónap jótállás vonatkozik."},
]
chunks = [{"id": d["id"], "text": d["text"], "metadata": {"source": d["source"]}} for d in documents]
chunks


In [ ]:
def simple_retrieve(question, chunks, top_k=2):
    terms = set(question.lower().split())
    scored = []
    for chunk in chunks:
        score = sum(1 for t in terms if t in chunk["text"].lower())
        scored.append((score, chunk))
    return [chunk for score, chunk in sorted(scored, reverse=True, key=lambda x: x[0])[:top_k]]

hits = simple_retrieve("Hány nap alatt érkezik meg Budapestre?", chunks)
hits


In [ ]:
allowed_tools = {"lookup_order": {"required": ["order_id"]}, "search_policy": {"required": ["query"]}}
model_plan = {"tool": "lookup_order", "arguments": {"order_id": "ORD-1042"}}

def validate_tool_call(plan):
    if plan["tool"] not in allowed_tools:
        raise ValueError("Unknown tool")
    missing = set(allowed_tools[plan["tool"]]["required"]) - set(plan["arguments"])
    if missing:
        raise ValueError(f"Missing args: {missing}")
    return True

validate_tool_call(model_plan)


In [ ]:
bench = [
    {"model": "3b-q4", "ttft_ms": 280, "tok_s": 42, "ram_gb": 3.2, "quality": 0.74},
    {"model": "8b-q4", "ttft_ms": 520, "tok_s": 24, "ram_gb": 6.1, "quality": 0.86},
    {"model": "14b-q4", "ttft_ms": 950, "tok_s": 14, "ram_gb": 9.8, "quality": 0.91},
]
bench


In [ ]:
def route_request(question, contains_pii=False, complexity=0.3, user_tier="standard"):
    if contains_pii:
        return "local-private"
    if complexity > 0.75:
        return "cloud-frontier"
    if user_tier == "free":
        return "local-small"
    return "local-balanced"

route_request("Elemezd ezt a panaszt", contains_pii=True, complexity=0.8)


In [ ]:
# Repo gyökeréből futtasd terminálban:
# docker compose --profile local-llm up -d
# curl http://localhost:11434/api/tags
lab_profile_command = "docker compose --profile local-llm up -d"
lab_profile_command


In [ ]:
eval_results = [
    {"provider": "local-8b", "grounded": 0.86, "policy": 0.94, "latency_ms": 1450, "cost": 0.00},
    {"provider": "cloud-frontier", "grounded": 0.94, "policy": 0.97, "latency_ms": 2100, "cost": 0.018},
]

for r in eval_results:
    r["pass"] = r["grounded"] >= 0.85 and r["policy"] >= 0.90 and r["latency_ms"] < 2500
eval_results
